# Lab 05: Python สำหรับ Statistical Learning
> CLO2 | LLo: อธิบายกรอบแนวคิด Statistical Learning แยกแยะ Supervised/Unsupervised และ Regression/Classification ได้

## บทนำ

ใน Lab นี้เราจะฝึกใช้ Python เพื่อสำรวจข้อมูลและประยุกต์ใช้กรอบ Statistical Learning ที่เรียนในบรรยาย เราจะใช้ **NumPy** จัดการ array ตัวเลข, **Pandas** อ่านและสำรวจ DataFrame, **Matplotlib/Seaborn** สร้าง visualization และ **scikit-learn** สร้าง model แรก Dataset หลักที่ใช้คือ **Advertising dataset** จาก ISLP — มี 200 ตลาด (observations) โดยแต่ละตลาดมีค่าใช้จ่ายโฆษณาทาง TV, Radio, Newspaper และยอดขาย (Sales) เป้าหมายคือให้นักศึกษาเห็น end-to-end workflow: Load → Inspect → Visualize → Model ซึ่งเป็นรูปแบบที่จะใช้ซ้ำตลอดวิชา เมื่อจบ Lab นี้แล้ว นักศึกษาจะสามารถ ระบุได้ว่าชุดข้อมูลใดเหมาะกับ problem type ใด และเริ่มต้น fit model ด้วย sklearn ได้


In [ ]:
# ─── Import libraries ────────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ที่ต้องใช้ทั้งหมดใน Lab 05
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ตั้งค่า plot ให้สวยงาม
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')


## Part 1: NumPy — Array Operations & Broadcasting

**Part นี้เราจะฝึกการดำเนินการกับ array เพื่อให้เข้าใจว่า data ใน machine learning คือ matrix ของตัวเลข**

NumPy เป็นรากฐานของ Python ทางวิทยาศาสตร์ — ทุก library (Pandas, sklearn, TensorFlow) ล้วนสร้างบน NumPy array ภายใน การเข้าใจ array operations จะช่วยให้ debug code ได้เร็วขึ้นมาก แนวคิดสำคัญที่ต้องจำ: ใน SL, X คือ matrix ขนาด n×p (n observations, p features) และ y คือ vector ขนาด n


In [ ]:
# ─── NumPy Array Operations ──────────────────────────────────────────
# วัตถุประสงค์: แสดงว่า X (feature matrix) และ y (response vector) คืออะไรใน SL

# สร้าง feature matrix X ขนาด 5×3 (5 observations, 3 features: TV, Radio, Newspaper)
X = np.array([
    [230.1, 37.8, 69.2],
    [44.5,  39.3, 45.1],
    [17.2,  45.9, 69.3],
    [151.5, 41.3, 58.5],
    [180.8, 10.8, 58.4]
])

# response vector y (Sales)
y = np.array([22.1, 10.4, 9.3, 18.5, 12.9])

print('Feature Matrix X (TV, Radio, Newspaper):')
print(X)
print(f'\nShape ของ X: {X.shape}  →  {X.shape[0]} observations, {X.shape[1]} features')
print(f'\nResponse Vector y (Sales): {y}')
print(f'Shape ของ y: {y.shape}')

# Broadcasting: คำนวณ standardized features (z-score)
X_mean = X.mean(axis=0)   # mean ของแต่ละ feature
X_std  = X.std(axis=0)    # std ของแต่ละ feature
X_scaled = (X - X_mean) / X_std   # broadcasting: ลบ vector จาก matrix ทีละ column

print('\nX หลัง standardize (mean=0, std=1):')
print(X_scaled.round(3))
print(f'\nMean หลัง scale: {X_scaled.mean(axis=0).round(6)}  (ควรเป็น ~0)')


### TODO 1: คำนวณ Dot Product สำหรับ Linear Prediction (Easy)

เราต้องการคำนวณ prediction ด้วย linear model เพราะ prediction ใน parametric SL คือ matrix multiplication

ให้คุณเขียน code คำนวณ $\hat{y} = X \cdot \beta$ โดยกำหนดให้ $\beta = [0.05, 0.18, -0.001]$ (coefficients สำหรับ TV, Radio, Newspaper) และ intercept = 2.9

**ผลลัพธ์ที่คาดหวัง**: array ของ predicted Sales สำหรับ 5 observations


In [ ]:
# TODO 1: เขียน code ที่นี่
# Hint: ใช้ np.dot() หรือ @ operator สำหรับ matrix multiplication
# Hint: prediction = X @ beta + intercept

beta = np.array([0.05, 0.18, -0.001])
intercept = 2.9

# คำนวณ y_hat ที่นี่
# y_hat = ...

raise NotImplementedError('กรุณาเติม code: คำนวณ y_hat = X @ beta + intercept')


## Part 2: Pandas — สำรวจ Advertising Dataset

**Part นี้เราจะโหลดและสำรวจ Advertising dataset เพื่อเข้าใจข้อมูลก่อนสร้าง model**

ขั้นตอน Inspect ข้อมูลเป็นสิ่งที่ Data Scientist ทำก่อนเสมอ — เพราะ 'garbage in, garbage out' ถ้าไม่เข้าใจข้อมูล model ที่ได้ก็ไร้ประโยชน์ เราจะใช้ Pandas functions หลัก: head(), info(), describe(), value_counts()


In [ ]:
# ─── สร้าง Advertising Dataset (mock ถ้าไม่มีไฟล์) ──────────────────
# วัตถุประสงค์: โหลดข้อมูลจริง หรือสร้าง mock dataset ที่มีโครงสร้างเดียวกัน
import os

# ลองโหลดจากไฟล์ก่อน ถ้าไม่มีให้สร้าง mock
DATA_PATH = r'E:\2569\Math for DS\Resources\Advertising.csv'

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    print('โหลดจากไฟล์จริงสำเร็จ!')
else:
    # สร้าง mock Advertising dataset (200 observations)
    np.random.seed(42)
    n = 200
    TV        = np.random.uniform(0.7, 296.4, n)
    Radio     = np.random.uniform(0.0, 49.6, n)
    Newspaper = np.random.uniform(0.3, 114.0, n)
    Sales     = (2.9 + 0.046*TV + 0.189*Radio - 0.001*Newspaper
                 + np.random.normal(0, 1.5, n))
    df = pd.DataFrame({'TV': TV, 'Radio': Radio,
                        'Newspaper': Newspaper, 'Sales': Sales})
    print('สร้าง mock Advertising dataset (200 obs) สำเร็จ!')

print(f'\nขนาด dataset: {df.shape[0]} rows × {df.shape[1]} columns')


In [ ]:
# ─── Inspect Dataset ──────────────────────────────────────────────────
# วัตถุประสงค์: เข้าใจโครงสร้างและลักษณะพื้นฐานของข้อมูลก่อน model

print('=== head() — 5 rows แรก ===')
display(df.head())

print('\n=== info() — ประเภทข้อมูลและ missing values ===')
df.info()

print('\n=== describe() — descriptive statistics ===')
display(df.describe().round(2))

# ตรวจ missing values
print('\n=== Missing Values ===')
print(df.isnull().sum())


## Part 3: Visualization — เห็น Pattern ใน Data

**Part นี้เราจะสร้าง visualization เพื่อเข้าใจความสัมพันธ์ระหว่าง features กับ Sales**

ก่อนสร้าง model ควรดูข้อมูลด้วยตาเสมอ — scatter plot ช่วยบอกว่าความสัมพันธ์เป็น linear หรือ non-linear histogram ช่วยบอก distribution ว่า skewed หรือ normal ข้อมูลเหล่านี้บอกว่าควรใช้ method แบบใด


In [ ]:
# ─── Scatter Plots: Feature vs Sales ────────────────────────────────
# วัตถุประสงค์: ดูความสัมพันธ์ระหว่าง input features กับ response Y

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

features = ['TV', 'Radio', 'Newspaper']
colors   = ['#2196F3', '#4CAF50', '#FF5722']

for ax, feat, col in zip(axes, features, colors):
    ax.scatter(df[feat], df['Sales'], alpha=0.5, color=col, edgecolors='white', linewidth=0.5)
    ax.set_xlabel(f'{feat} (พัน USD)', fontsize=12)
    ax.set_ylabel('Sales (พันหน่วย)', fontsize=12)
    ax.set_title(f'{feat} vs Sales', fontsize=13)
    # คำนวณ correlation
    r = df[feat].corr(df['Sales'])
    ax.text(0.05, 0.95, f'r = {r:.3f}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Advertising Dataset: Features vs Sales', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nสังเกต: TV มี correlation สูงสุดกับ Sales → TV น่าจะเป็น predictor ที่สำคัญ')


In [ ]:
# ─── Distribution Plots ───────────────────────────────────────────────
# วัตถุประสงค์: ดู distribution ของแต่ละ variable เพื่อตรวจ skewness และ outliers

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=25, color='#5C6BC0', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    # เพิ่ม mean line
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean={df[col].mean():.1f}')
    axes[i].legend()

plt.suptitle('Distribution ของแต่ละ Variable', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### TODO 2: สร้าง Pair Plot และ Correlation Heatmap (Medium)

เราต้องการเห็นความสัมพันธ์ระหว่าง **ทุก** คู่ของ variables พร้อมกัน เพราะการดู scatter plot ทีละคู่ใช้เวลานาน

ให้คุณ: (1) สร้าง pair plot ด้วย `sns.pairplot()` และ (2) สร้าง correlation heatmap ด้วย `sns.heatmap()` พร้อม annotate ค่า r

**ผลลัพธ์ที่คาดหวัง**: pair plot แสดง scatter ทุกคู่, heatmap แสดง correlation matrix ที่มีตัวเลขในแต่ละ cell


In [ ]:
# TODO 2: สร้าง Pair Plot และ Correlation Heatmap
# Hint สำหรับ pair plot: sns.pairplot(df, diag_kind='hist')
# Hint สำหรับ heatmap: sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='coolwarm')

# Part 2a: Pair Plot
# pair_plot = ...

# Part 2b: Correlation Heatmap
# corr_matrix = df.corr()
# sns.heatmap(...)

raise NotImplementedError('กรุณาเติม code: pair plot และ correlation heatmap')


## Part 4: scikit-learn — Fit Your First Model

**Part นี้เราจะใช้ sklearn API (fit → predict → score) เพื่อสร้าง model แรกจาก Advertising data**

sklearn ใช้ pattern เดียวกันสำหรับทุก algorithm: สร้าง estimator → fit() → predict() เมื่อเข้าใจ pattern นี้แล้ว การเรียนรู้ algorithm ใหม่จะง่ายมาก เพียงเปลี่ยน class เท่านั้น เราจะเริ่มจาก Linear Regression (parametric) และ KNN Regression (non-parametric)


In [ ]:
# ─── Fit Linear Regression (Parametric) ─────────────────────────────
# วัตถุประสงค์: เห็น sklearn API และเปรียบเทียบกับ parametric SL framework

# เตรียม X และ y
X_data = df[['TV', 'Radio', 'Newspaper']].values   # n×3 matrix
y_data = df['Sales'].values                          # n-vector

# แบ่ง Train (80%) / Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape[0]} obs, Test: {X_test.shape[0]} obs')

# สร้างและ fit Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)          # ประมาณ beta จาก training data

# ผล: estimated f̂ (parametric)
print('\n=== Linear Regression Results ===')
print(f'Intercept (β₀): {lr.intercept_:.4f}')
for name, coef in zip(['TV', 'Radio', 'Newspaper'], lr.coef_):
    print(f'β_{name}: {coef:.4f}')

# Predict และ evaluate
y_pred_lr = lr.predict(X_test)
mse_lr    = mean_squared_error(y_test, y_pred_lr)
r2_lr     = lr.score(X_test, y_test)

print(f'\nTest MSE = {mse_lr:.4f}')
print(f'Test R²  = {r2_lr:.4f}')
print(f'\nInterpretation: β_TV = {lr.coef_[0]:.4f} หมายความว่า')
print(f'  เพิ่มงบ TV อีก 1 พัน USD → Sales เพิ่ม {lr.coef_[0]:.4f} พันหน่วย')


In [ ]:
# ─── Fit KNN Regression (Non-parametric) ─────────────────────────────
# วัตถุประสงค์: เปรียบเทียบ non-parametric กับ parametric approach

# Fit KNN Regression ด้วย K=5
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
mse_knn    = mean_squared_error(y_test, y_pred_knn)
r2_knn     = knn.score(X_test, y_test)

print('=== KNN Regression (K=5) Results ===')
print(f'Test MSE = {mse_knn:.4f}')
print(f'Test R²  = {r2_knn:.4f}')

# เปรียบเทียบ
print('\n=== เปรียบเทียบ Linear vs KNN ===')
print(f'{"Method":<20} {"MSE":<12} {"R²"}')
print('-' * 40)
print(f'{"Linear Regression":<20} {mse_lr:<12.4f} {r2_lr:.4f}')
print(f'{"KNN (K=5)":<20} {mse_knn:<12.4f} {r2_knn:.4f}')
print('\nNote: ตัวไหน MSE ต่ำกว่า = ดีกว่าสำหรับ prediction')


### TODO 3: วิเคราะห์ Effect of K ใน KNN (Hard)

เราต้องการเข้าใจว่า K ส่งผลต่อ accuracy ของ KNN อย่างไร เพราะนี่คือ Bias-Variance trade-off แรกที่จะเห็น

ให้คุณ: (1) ทดลอง K ตั้งแต่ 1 ถึง 30 ทีละ 1 (2) คำนวณ Train MSE และ Test MSE สำหรับแต่ละ K (3) plot กราฟ Train MSE vs Test MSE เทียบกับ K

**ผลลัพธ์ที่คาดหวัง**: กราฟที่เห็นว่า K เล็ก → overfitting (train MSE ต่ำมากแต่ test MSE สูง) และ K ใหญ่ → underfitting (ทั้ง train และ test MSE สูง)

**คำถาม reflection**: K ที่ดีที่สุดในชุดข้อมูลนี้คือเท่าไร? เพราะอะไร?


In [ ]:
# TODO 3: วิเคราะห์ Effect of K
# Hint: ใช้ loop: for k in range(1, 31): fit KNN(k), compute train/test MSE
# Hint: เก็บใน lists แล้ว plot ด้วย plt.plot()

k_values = range(1, 31)
train_mses = []
test_mses  = []

# เขียน loop ที่นี่
# for k in k_values:
#     model = ...
#     train_mses.append(...)
#     test_mses.append(...)

# สร้าง plot
# plt.figure(figsize=(10, 6))
# plt.plot(k_values, train_mses, label='Train MSE')
# plt.plot(k_values, test_mses,  label='Test MSE')
# ...

raise NotImplementedError('กรุณาเติม code: วิเคราะห์ effect of K')


## Reflection Questions

ตอบคำถามต่อไปนี้ใน Markdown cell นี้:

**Q1**: จาก scatter plots ใน Part 3 ให้บอกว่า feature ใดน่าจะ useful ที่สุดสำหรับ predict Sales และ feature ใดน่าจะ least useful — อธิบายเหตุผล

**Q2**: เปรียบเทียบ Linear Regression กับ KNN (K=5) — ถ้าโจทย์ต้องการ *inference* (ต้องการรู้ว่า feature ใดสำคัญ) ควรเลือกใช้ method ใด? เพราะอะไร?

**Q3**: จาก TODO 3 ค่า K ที่ให้ Test MSE ต่ำสุดคือเท่าไร? ถ้า K=1 Train MSE เป็นเท่าไร? อธิบายปรากฏการณ์นี้ในบริบทของ Bias-Variance Trade-Off

*[พิมพ์คำตอบที่นี่]*
